In [115]:
#imports
import pandas as pd
import numpy as np







In [ ]:
### Men's Massey Ordinals

#data
massey_ordinals_m = pd.read_csv("MarchMachineLearningMania2026/data_2025/MMasseyOrdinals.csv")

#pre tournament rankings
pre_tournament_massey_m = massey_ordinals_m.query("RankingDayNum == 133")

#good subset of rankings
good_ratings_m = ["POM", "EBP", "MAS", "TRK", "HAS"]
g_pre_tournament_massey_m = massey_ordinals_m.query("Season >= 2016").loc[lambda df: df["SystemName"].isin(good_ratings_m)]

#group by and summarize
avg_g_pre_tournament_massey_m = g_pre_tournament_massey_m.groupby(["Season", "TeamID"]).agg(
    avg_rank=("OrdinalRank", "mean")
)


In [117]:
### Men's Stats

#data
reg_stats_m = pd.read_csv("MarchMachineLearningMania2026/data_2025/MRegularSeasonDetailedResults.csv")

#Remove Loc, which causes problems
reg_stats_m = reg_stats_m.drop(columns = 'WLoc')

w_reg_stats_m = reg_stats_m.copy()
l_reg_stats_m = reg_stats_m.copy()

#For games team is winner
w_reg_stats_m.columns = w_reg_stats_m.columns.str.replace(r'^L', 'opp_', regex=True)
w_reg_stats_m.columns = w_reg_stats_m.columns.str.replace(r'^W', '', regex=True)

#For games team is loser
l_reg_stats_m.columns = l_reg_stats_m.columns.str.replace(r'^W', 'opp_', regex=True)
l_reg_stats_m.columns = l_reg_stats_m.columns.str.replace(r'^L', '', regex=True)

#combine the data
reg_stats_pg_m = pd.concat([w_reg_stats_m, l_reg_stats_m])

#per game percentages (need for tempo adjusted average percentages)
reg_stats_pg_m = (
    reg_stats_pg_m.assign(margin = reg_stats_pg_m['Score'] - reg_stats_pg_m['opp_Score'])
        .assign(poss = lambda x: x['FGA'] - x['OR'] + x['TO'] + 0.475*x['FTA'])
        .assign(opp_poss = lambda x: x['opp_FGA'] - x['opp_OR'] + x['opp_TO'] + 0.475*x['opp_FTA'])
        .assign(eff=lambda x: x['Score'] / x['poss'])
        .assign(opp_eff=lambda x: x['opp_Score'] / x['opp_poss'])

        .assign(fg_per=lambda x: x['FGM'] / x['FGA'])
        .assign(fg_a_per=lambda x: x['FGA'] / x['poss'])
        .assign(thr_a_per=lambda x: x['FGA3'] / x['poss'])
        .assign(to_per=lambda x: x['TO'] / x['poss'])
        .assign(blk_per=lambda x: x['Blk'] / x['opp_FGA'])
        .assign(foul_rec_per=lambda x: x['opp_PF'] / x['poss'])
        .assign(foul_per=lambda x: x['PF'] / x['opp_poss'])
        .assign(or_per=lambda x: x['OR'] / (x['OR'] + x['opp_DR']))
        .assign(dr_per=lambda x: x['DR'] / (x['DR'] + x['opp_OR']))
        
        .assign(opp_fg_a_per=lambda x: x['opp_FGA'] / x['opp_poss'])
        .assign(opp_fg_per=lambda x: x['opp_FGM'] / x['opp_FGA'])
        .assign(opp_to_per=lambda x: x['opp_TO'] / x['opp_poss'])
)

f_reg_stats_m = reg_stats_pg_m.groupby(["Season", "TeamID"]).agg(
    avg_score=("Score", "mean"),
    avg_opp_score=("opp_Score", "mean"),
    avg_margin = ("margin", "mean"),
    avg_poss = ("poss", "mean"),
    avg_eff = ("eff", "mean"),
    avg_opp_eff = ("opp_eff", "mean"),
    avg_fg_per=("fg_per", "mean"),
    avg_m3=("FGM3", "mean"),
    avg_a3=("FGA3", "mean"),#
    avg_ftm=("FTM", "mean"),
    avg_fta=("FTA", "mean"),#
    avg_fg_a_per=("fg_a_per", "mean"),
    avg_thr_a_per=("thr_a_per", "mean"),
    avg_to_per=("to_per", "mean"),
    avg_blk_per=("blk_per", "mean"),
    avg_foul_rec_per=("foul_rec_per", "mean"),
    avg_foul_per=("foul_per", "mean"),
    avg_or_per=("or_per", "mean"),
    avg_dr_per=("dr_per", "mean"),
    avg_opp_fg_a_per=("opp_fg_a_per", "mean"),
    avg_opp_fg_per=("opp_fg_per", "mean"),
    avg_opp_to_per=("opp_to_per", "mean")
)

f_reg_stats_m = (
    f_reg_stats_m.assign(avg_thr_per = f_reg_stats_m['avg_m3'] / f_reg_stats_m['avg_a3'])
      .assign(ft_per=lambda x: x['avg_ftm'] / x['avg_fta'])
)

f_reg_stats_m = f_reg_stats_m.drop(columns = ['avg_a3', 'avg_m3', 'avg_ftm', 'avg_fta'])

f_reg_stats_m.sort_values('avg_eff', ascending = False).head()

,,avg_score,avg_opp_score,avg_margin,avg_poss,avg_eff,avg_opp_eff,avg_fg_per,avg_fg_a_per,avg_thr_a_per,avg_to_per,avg_blk_per,avg_foul_rec_per,avg_foul_per,avg_or_per,avg_dr_per,avg_opp_fg_a_per,avg_opp_fg_per,avg_opp_to_per,avg_thr_per,ft_per
Season,TeamID,,,,,,,,,,,,,,,,,,,,
2019,1211,88.848485,65.060606,23.787879,71.668182,1.237302,0.908231,0.530947,0.846184,0.297936,0.144485,0.089764,0.260265,0.224976,0.301536,0.731357,0.860610,0.387227,0.188974,0.365057,0.767409
2018,1437,87.058824,70.882353,16.176471,70.868382,1.229814,0.997585,0.505257,0.871187,0.407042,0.145431,0.065018,0.242710,0.222384,0.283763,0.736646,0.848740,0.435525,0.184785,0.397949,0.771285
2025,1181,82.705882,61.911765,20.794118,67.393382,1.229000,0.913745,0.488629,0.877097,0.398971,0.136664,0.065625,0.244543,0.234826,0.335700,0.771748,0.853604,0.387337,0.158629,0.377193,0.784496
2024,1163,81.470588,64.411765,17.058824,66.952941,1.214034,0.952506,0.496698,0.876202,0.357518,0.136418,0.094713,0.251812,0.240121,0.348633,0.768780,0.835231,0.398781,0.147549,0.366871,0.742470
2014,1166,79.545455,67.424242,12.121212,65.557576,1.212873,1.034567,0.500719,0.842861,0.376917,0.150976,0.024571,0.283724,0.270288,0.274369,0.722678,0.868145,0.422950,0.149718,0.421182,0.751603
